# Ingesta reproducible del dataset Adult

Equivalente explicado de `ingest.py`. El propósito es descargar desde la fuente oficial y conservar una copia *raw* sin aplicar limpieza ni feature engineering. La separación entre ingesta y transformación permite rastrear el origen de los datos.

## 1. Preparar rutas
La ruta se resuelve desde la raíz del proyecto. Así el notebook funciona abierto desde la raíz o desde `notebooks/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_PATH = PROJECT_ROOT / 'adult_raw.csv'
OUTPUT_PATH

WindowsPath('c:/Users/c3283/Desktop/INCOEX/repositorio24-08-2026/adult_raw.csv')

## 2. Descargar desde UCI
`ucimlrepo` identifica Adult con el ID 2. La descarga requiere conexión a Internet. No se usa un CSV preparado manualmente.

In [2]:
from ucimlrepo import fetch_ucirepo
adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y = adult.data.targets.copy()
X.shape, y.shape

((48842, 14), (48842, 1))

## 3. Unir features y target
La fuente entrega ambos objetos por separado. Se reconstruye una tabla única sin modificar valores.

In [3]:
data = X.copy()
data[y.columns[0]] = y.iloc[:, 0]
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 4. Validación básica de ingesta
Estas comprobaciones no sustituyen los Data Quality Gates. Solo verifican que la descarga produjo una tabla utilizable.

In [4]:
assert not data.empty, 'La descarga no produjo registros'
assert data.shape[1] == 15, 'Se esperaban 14 features y un target'
print(f'Registros: {data.shape[0]:,}')
print(f'Columnas: {data.shape[1]}')
data.dtypes

Registros: 48,842
Columnas: 15


age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object

## 5. Guardar la capa raw
Al ejecutar esta celda se reemplaza `adult_raw.csv` por una nueva descarga oficial. La limpieza posterior ocurre en `src/data.py`, no aquí.

In [5]:
data.to_csv(OUTPUT_PATH, index=False)
print(f'Dataset raw guardado en: {OUTPUT_PATH}')

Dataset raw guardado en: c:\Users\c3283\Desktop\INCOEX\repositorio24-08-2026\adult_raw.csv


## Decisión técnica
La ingesta queda aislada y reproducible. Para automatización se mantiene `python ingest.py`; este notebook sirve como explicación y evidencia ejecutable del mismo proceso.